# Aesthetic score example


In [ ]:
import sys
import os
if ".." not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))

import torch
from utils.color.linear_to_srgb_converters import LinearRec709ToAgXBase, SimpleGammaCurve
from utils.color.tonemapping.agx_looks import AgXPunchyLook
from utils.losses.aesthetic import LAIONAestheticScorer, AestheticLossMaximize
from utils.train import train_with_criterion
from scenes import CarScene, SpringPortraitSmallDomeScene, HouseScene, SciFiRobotScene

In [ ]:
torch_precision = torch.float32
device = 'cuda' if torch.cuda.is_available() else 'cpu'

lr = 0.06
n_iter = 900
global_seed = 3  # Can be None
n_results = 3

output_subdirectory_name = 'aesthetic_score_example'

In [ ]:
scene = SciFiRobotScene()
# color_space_converter = SimpleGammaCurve()
color_space_converter = LinearRec709ToAgXBase(AgXPunchyLook())
scorer = LAIONAestheticScorer(device=device)
criterion = AestheticLossMaximize(scorer)
title_prefix = f'Aesthetic Maximize - {scene.name}'

In [ ]:
train_with_criterion(
    scene,
    lr, n_iter, criterion,
    starting_multiplier_std=(0.2, 0.2, 0.2),
    starting_multiplier_mean=1.0,
    output_subdirectory_name=output_subdirectory_name,
    n_results=n_results,
    torch_precision=torch_precision,
    render_color_space_converter=color_space_converter,
    require_physically_plausible_multipliers=True,
    title_prefix=title_prefix,
    device=device,
    save_every=75,
    model_name='LAIONAesthetic_vit_b_32',
    pretrained_source='LAION sa_0_4_vit_b_32_linear',
    seed=global_seed,
    run_name_suffix=scene.name,
)